# Phase 3 & 4: Modelling — Logistic Regression + XGBoost

**Goal:** Train a logistic regression baseline, then train XGBoost, compare both on held-out test data, and save the best model.

In [ ]:
import sys
sys.path.append('..')

import pickle
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import mlflow

from src.data.loader import load_raw
from src.features.engineer import engineer, split
from src.models.train import train_logistic, train_xgboost
from src.models.evaluate import report

sns.set_theme(style='whitegrid', palette='muted')

MODELS_DIR = Path('../models')
MODELS_DIR.mkdir(exist_ok=True)

## 1. Load & Prepare Data

In [ ]:
df = engineer(load_raw())
X_train, X_test, y_train, y_test = split(df)

print(f"Train: {X_train.shape} | Test: {X_test.shape}")
print(f"Positive class rate — train: {y_train.mean():.4f} | test: {y_test.mean():.4f}")

## 2. Logistic Regression (Baseline)

`class_weight='balanced'` tells sklearn to up-weight the minority class (defaults) so the model does not ignore them.

In [ ]:
mlflow.set_experiment('credit-risk-model')

with mlflow.start_run(run_name='logistic_regression'):
    lr = train_logistic(X_train, y_train)
    lr_prob = lr.predict_proba(X_test)[:, 1]
    lr_metrics = report(y_test, lr_prob)
    mlflow.log_params({'model': 'logistic_regression', 'class_weight': 'balanced'})
    mlflow.log_metrics(lr_metrics)

print('Logistic Regression metrics:')
for k, v in lr_metrics.items():
    print(f'  {k}: {v}')

## 3. XGBoost

`scale_pos_weight` = (# negatives) / (# positives) — equivalent to `class_weight='balanced'` but native to XGBoost. Tells the booster to penalise missing a default more than missing a non-default.

In [ ]:
with mlflow.start_run(run_name='xgboost'):
    xgb = train_xgboost(X_train, y_train)
    xgb_prob = xgb.predict_proba(X_test)[:, 1]
    xgb_metrics = report(y_test, xgb_prob)
    mlflow.log_params({'model': 'xgboost', 'n_estimators': 300,
                       'learning_rate': 0.05, 'max_depth': 6})
    mlflow.log_metrics(xgb_metrics)

print('XGBoost metrics:')
for k, v in xgb_metrics.items():
    print(f'  {k}: {v}')

## 4. Model Comparison

In [ ]:
comparison = pd.DataFrame(
    [lr_metrics, xgb_metrics],
    index=['Logistic Regression', 'XGBoost']
)
comparison.style.highlight_max(axis=0, color='#d4edda')

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
comparison.plot(kind='bar', ax=ax, edgecolor='white', width=0.6)
ax.set_title('Model Comparison — ROC-AUC, KS, Gini', fontsize=13)
ax.set_ylabel('Score')
ax.set_ylim(0, 1)
ax.tick_params(axis='x', rotation=0)
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

## 5. Save Best Model (XGBoost)

In [ ]:
with open(MODELS_DIR / 'xgboost.pkl', 'wb') as f:
    pickle.dump(xgb, f)

print('Model saved → models/xgboost.pkl')
print(f'\nMLflow UI: run `mlflow ui` then open http://localhost:5000')